In [ ]:
import pandas as pd
import os

d = 10

# Input CSV files
csv_msg = f"../generated_msg/msg_ela/ela_msg_{d}d_all.csv"
csv_bbob = f"../generated_msg/bbob_ela/ela_bbob_{d}d_all.csv"

# Output CSV
output_csv = f"../generated_msg/combined_{d}d_all.csv"
# Load CSVs
df1 = pd.read_csv(csv_msg)
df2 = pd.read_csv(csv_bbob)

# Optional: check columns match
if list(df1.columns) != list(df2.columns):
    raise ValueError("CSV columns do not match!")

# Concatenate rows
df_combined = pd.concat([df1, df2], axis=0, ignore_index=True)

# Save combined CSV
os.makedirs(os.path.dirname(output_csv), exist_ok=True)
df_combined.to_csv(output_csv, index=False)

print(f"[SAVED] Combined CSV: {output_csv}")


In [ ]:
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Function to compute seed noise ratio for a single CSV
# -------------------------------
def culc_seed_noise_ratio_single(csv_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    df_name = os.path.splitext(os.path.basename(csv_path))[0]
    print(f"[INFO] Processing {csv_path}")

    # Load CSV
    df = pd.read_csv(csv_path)

    # Feature columns
    feature_cols = [c for c in df.columns if c not in ["function_id", "instance_id", "seed"]]

    # Standardize features
    scaler = StandardScaler()
    df[feature_cols] = scaler.fit_transform(df[feature_cols])

    results = {}
    for feat in feature_cols:
        total_var = df[feat].var(ddof=0)
        seed_var = df.groupby(["function_id", "instance_id"])[feat].var(ddof=0)
        mean_seed_var = seed_var.mean()

        results[feat] = {
            "mean_seed_var": mean_seed_var,
            "total_var": total_var,
            "ratio": mean_seed_var / total_var if total_var > 0 else 1.0
        }

    # Save results
    df_results = pd.DataFrame.from_dict(results, orient="index")
    df_results.index.name = "feature"
    df_results.reset_index(inplace=True)
    output_path = os.path.join(output_dir, f"{df_name}_seed_noise_ratio.csv")
    df_results.to_csv(output_path, index=False)
    print(f"[SAVED] {output_path}")

# -------------------------------
# Paths
# -------------------------------
csv_file = f"../generated_msg/combined_{d}d_all.csv"       # Single CSV
output_dir = f"../generated_msg"

# -------------------------------
# Run
# -------------------------------
culc_seed_noise_ratio_single(csv_file, output_dir)


In [ ]:
import pandas as pd
import os

# -------------------------------
# Parameters
# -------------------------------
input_csv = f"../generated_msg/combined_{d}d_all_seed_noise_ratio.csv"
out_dir = f"../generated_msg"
os.makedirs(out_dir, exist_ok=True)

# -------------------------------
# Load seed noise ratio results
# -------------------------------
df_results = pd.read_csv(input_csv)

# -------------------------------
# Set threshold (Q3) for unstable features
# -------------------------------
threshold = df_results["ratio"].quantile(0.75)

# Separate stable and unstable features
unstable_features = df_results.loc[df_results["ratio"] >= threshold, "feature"].tolist()
stable_features = df_results.loc[df_results["ratio"] < threshold, "feature"].tolist()

# -------------------------------
# Print summary
# -------------------------------
print(f"Threshold (Q3): {threshold:.4f}")
print(f"Stable features ({len(stable_features)}): {stable_features}")
print(f"Unstable features ({len(unstable_features)}): {unstable_features}")

# -------------------------------
# Save to CSV
# -------------------------------
pd.DataFrame({"feature": stable_features}).to_csv(
    os.path.join(out_dir, f"stable_s_ela_features_{d}d.csv"), index=False
)
pd.DataFrame({"feature": unstable_features}).to_csv(
    os.path.join(out_dir, f"unstable_s_ela_features_{d}d.csv"), index=False
)

print("[SAVED] Stable/unstable feature CSVs")


In [ ]:
import pandas as pd
import os

# -------------------------------
# Parameters
# -------------------------------
ref_dir = "../generated_msg"         # Folder containing original CSV
out_dir = "../generated_msg"         # Output folder
os.makedirs(out_dir, exist_ok=True)

# -------------------------------
# Load combined CSV
# -------------------------------
csv_path = f"{ref_dir}/combined_{d}d_all.csv"
df = pd.read_csv(csv_path)

# -------------------------------
# Load stable feature list
# -------------------------------
stable_features_path = f"{out_dir}/stable_s_ela_features_{d}d.csv"
stable_features_df = pd.read_csv(stable_features_path)
stable_features = stable_features_df["feature"].tolist()

# -------------------------------
# Keep identifiers + stable features
# -------------------------------
cols_to_keep = ["function_id", "instance_id", "seed"] + stable_features
df_stable = df[cols_to_keep]

# -------------------------------
# Save filtered CSV
# -------------------------------
out_path = f"{out_dir}/combined_{d}d_all_stable.csv"
df_stable.to_csv(out_path, index=False)

print(f"[SAVED] Stable features CSV: {out_path}")


In [ ]:
import pandas as pd
import os

# -------------------------------
# Parameters
# -------------------------------
ref_dir = "../generated_msg"       # Folder with stable CSV
out_dir = "../generated_msg"       # Output folder
os.makedirs(out_dir, exist_ok=True)

# -------------------------------
# Load stable features CSV
# -------------------------------
csv_path = f"{ref_dir}/combined_{d}d_all_stable.csv"
df = pd.read_csv(csv_path)

# -------------------------------
# Compute median across seeds
# -------------------------------
# Group by function_id and instance_id and compute median
df_median = df.groupby(["function_id", "instance_id"], as_index=False).median(numeric_only=True)

# -------------------------------
# Save median CSV
# -------------------------------
out_path = f"{out_dir}/combined_{d}d_median_stable.csv"
df_median.to_csv(out_path, index=False)

print(f"[SAVED] Median stable features CSV: {out_path}")
